# 6. XGBoost

**What it does:** builds decision trees one after another, with each new tree correcting the mistakes of the earlier ones.

**Why try it:** it often does very well on tables of numbers.

**Target:** referable DR, meaning grade 2 or worse.

Run `01_clean_and_eda.ipynb` and `02_image_features.ipynb` first.

In [1]:
%pip install -q numpy pandas scipy pillow matplotlib scikit-learn joblib xgboost

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\Users\\Administrator\\Desktop\\AfriEyeSamsung\\.venv\\Lib\\site-packages\\sklearn\\_cyutility.cp311-win_amd64.pyd'
Check the permissions.


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw" / "DeepDRiD" / "regular_fundus_images"
PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
MODELS = ROOT / "models"
for folder in (PROCESSED, RESULTS, MODELS):
    folder.mkdir(parents=True, exist_ok=True)
print("Project folder:", ROOT)

Project folder: c:\Users\Administrator\Desktop\AfriEyeSamsung


In [3]:
import matplotlib.pyplot as plt

BLUE, TEAL, ORANGE, PURPLE = "#1F4E9E", "#00948F", "#C4581F", "#7A3E9D"
INK, MUTED, GRID = "#1B2433", "#566173", "#E3E7F0"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110, "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.edgecolor": "#AEB6C5", "axes.labelcolor": MUTED, "axes.titlecolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID, "axes.axisbelow": True,
    "legend.frameon": False,
})

In [4]:
import json
import time

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import GroupKFold, RandomizedSearchCV, GridSearchCV

labels = pd.read_csv(PROCESSED / "clean_labels.csv")
features = pd.read_csv(PROCESSED / "features.csv")
table = labels.merge(features, on="image_id")
FEATURE_COLS = [c for c in features.columns if c != "image_id"]
train, val, test = [table[table.split == s].reset_index(drop=True) for s in ("train", "val", "test")]
print(f"Train {len(train)} photos, validation {len(val)}, test {len(test)}. {len(FEATURE_COLS)} features.")
CV = GroupKFold(n_splits=5)  # keeps each patient's photos together while tuning


def threshold_for_sensitivity(y, score, target=0.90):
    """Highest cut-off that still catches the target share of referable photos. Use validation data only."""
    positives = np.sort(np.asarray(score)[np.asarray(y) == 1])[::-1]
    return float(positives[max(0, int(np.ceil(target * len(positives))) - 1)])


def rates(y, score, threshold):
    tn, fp, fn, tp = confusion_matrix(y, np.asarray(score) >= threshold, labels=[0, 1]).ravel()
    return {"sensitivity": tp / max(1, tp + fn), "specificity": tn / max(1, tn + fp), "tp": int(tp), "fn": int(fn), "tn": int(tn), "fp": int(fp)}


def patient_ci(metric, frame, score, repeats=1000, seed=0):
    """95% interval. It resamples whole patients, because one patient's photos are related."""
    rng = np.random.default_rng(seed)
    groups = frame.groupby("patient_id").indices
    ids = list(groups)
    y, score = frame.referable.values, np.asarray(score)
    results = []
    for _ in range(repeats):
        take = np.concatenate([groups[i] for i in rng.choice(ids, len(ids))])
        if 0 < y[take].sum() < len(take):
            results.append(metric(y[take], score[take]))
    return [float(np.percentile(results, 2.5)), float(np.percentile(results, 97.5))]


def report(name, val_score, test_score, seconds, best_params, extra=None):
    threshold = threshold_for_sensitivity(val.referable, val_score)
    v, t = rates(val.referable, val_score, threshold), rates(test.referable, test_score, threshold)
    result = {
        "model": name, "seconds_to_train": seconds, "best_params": best_params, "threshold": threshold,
        "val_auc": roc_auc_score(val.referable, val_score), "val_sensitivity": v["sensitivity"], "val_specificity": v["specificity"],
        "test_auc": roc_auc_score(test.referable, test_score), "test_auc_ci": patient_ci(roc_auc_score, test, test_score),
        "test_sensitivity": t["sensitivity"], "test_sensitivity_ci": patient_ci(lambda y, s: rates(y, s, threshold)["sensitivity"], test, test_score),
        "test_specificity": t["specificity"], "test_specificity_ci": patient_ci(lambda y, s: rates(y, s, threshold)["specificity"], test, test_score),
        "test_confusion": {k: t[k] for k in ("tp", "fn", "tn", "fp")},
        "missed_severe_or_worse": int(((test.grade >= 3) & (np.asarray(test_score) < threshold)).sum()), "severe_or_worse": int((test.grade >= 3).sum()),
    }
    result.update(extra or {})
    print(f"Validation AUC {result['val_auc']:.3f}, threshold {threshold:.3f}")
    print(f"Test AUC         {result['test_auc']:.3f}  (95% CI {result['test_auc_ci'][0]:.3f} to {result['test_auc_ci'][1]:.3f})")
    print(f"Test sensitivity {result['test_sensitivity']:.1%}  (95% CI {result['test_sensitivity_ci'][0]:.1%} to {result['test_sensitivity_ci'][1]:.1%})")
    print(f"Test specificity {result['test_specificity']:.1%}  (95% CI {result['test_specificity_ci'][0]:.1%} to {result['test_specificity_ci'][1]:.1%})")
    print(f"Severe or proliferative photos missed: {result['missed_severe_or_worse']} of {result['severe_or_worse']}")
    return result


def result_charts(name, test_score, threshold, color):
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.7), gridspec_kw={"width_ratios": [1.1, .9, 1.1]})
    y = test.referable.values
    fpr, tpr, _ = roc_curve(y, test_score)
    t = rates(y, test_score, threshold)
    axes[0].plot([0, 1], [0, 1], color="#AEB6C5", linestyle="--", linewidth=1, label="Chance")
    axes[0].plot(fpr, tpr, color=color, linewidth=2, label=name)
    axes[0].plot(1 - t["specificity"], t["sensitivity"], "o", color=color, markersize=9, markeredgecolor="white", label="Chosen threshold")
    axes[0].set(xlabel="False positive rate", ylabel="Sensitivity", title=f"ROC curve, AUC {roc_auc_score(y, test_score):.2f}", xlim=(0, 1), ylim=(0, 1.02)); axes[0].legend(loc="lower right", fontsize=8)
    matrix = np.array([[t["tn"], t["fp"]], [t["fn"], t["tp"]]])
    axes[1].imshow(matrix, cmap="Blues", vmin=0, vmax=matrix.max() * 1.6)
    for (r, c), v in np.ndenumerate(matrix):
        axes[1].text(c, r, str(v), ha="center", va="center", fontsize=15, fontweight="bold", color=INK)
    axes[1].set_xticks([0, 1], ["Cleared", "Referred"]); axes[1].set_yticks([0, 1], ["Not referable", "Referable"], rotation=90, va="center"); axes[1].grid(False)
    axes[1].set_title("Results on test photos"); axes[1].spines[:].set_visible(False)
    bins = np.linspace(np.min(test_score), np.max(test_score), 26)
    axes[2].hist(np.asarray(test_score)[y == 0], bins=bins, color="#AEB6C5", label="Not referable")
    axes[2].hist(np.asarray(test_score)[y == 1], bins=bins, color=color, alpha=.82, label="Referable")
    axes[2].axvline(threshold, color=INK, linewidth=1.5, label="Threshold")
    axes[2].set(title="Scores by true group", xlabel="Model score", ylabel="Photos"); axes[2].legend(fontsize=8); axes[2].grid(axis="x", visible=False)
    fig.tight_layout(); plt.show()


def importance_chart(values, title, color, top=15):
    s = pd.Series(values, index=FEATURE_COLS).sort_values(key=np.abs, ascending=False).head(top)[::-1]
    fig, ax = plt.subplots(figsize=(8, 4.6))
    bars = ax.barh(s.index, s.values, color=color, height=.68)
    ax.bar_label(bars, fmt="%.3f", padding=3, color=INK, fontsize=8); ax.axvline(0, color="#AEB6C5", linewidth=1)
    ax.set_title(title); ax.grid(axis="y", visible=False); ax.margins(x=.14); fig.tight_layout(); plt.show()


def save_all(result, model, val_score, test_score, slug):
    (RESULTS / f"{slug}_metrics.json").write_text(json.dumps(result, indent=1))
    pd.concat([val.assign(score=val_score)[["image_id", "patient_id", "split", "grade", "referable", "score"]], test.assign(score=test_score)[["image_id", "patient_id", "split", "grade", "referable", "score"]]]).to_csv(RESULTS / f"{slug}_predictions.csv", index=False)
    joblib.dump(model, MODELS / f"{slug}.joblib")
    print("Saved:", f"results/{slug}_metrics.json, results/{slug}_predictions.csv, models/{slug}.joblib")

Train 100 photos, validation 0, test 0. 72 features.


## Train the model

We use only training photos, and we keep each patient's photos together while tuning.

In [5]:
from xgboost import XGBClassifier

start = time.time()
balance = (train.referable == 0).sum() / (train.referable == 1).sum()
settings = {"n_estimators": [200, 400, 700], "max_depth": [2, 3, 4, 6], "learning_rate": [0.01, 0.03, 0.06, 0.1], "subsample": [0.7, 0.85, 1.0], "colsample_bytree": [0.5, 0.75, 1.0], "min_child_weight": [1, 3, 6], "reg_lambda": [1, 3, 10]}
search = RandomizedSearchCV(XGBClassifier(tree_method="hist", eval_metric="auc", scale_pos_weight=balance, random_state=0, n_jobs=1), settings, n_iter=20, scoring="roc_auc", cv=CV, random_state=0, n_jobs=-1)
search.fit(train[FEATURE_COLS], train.referable, groups=train.patient_id)
model = search.best_estimator_
seconds = time.time() - start
print("Best settings:", search.best_params_)
print(f"Cross-validated AUC {search.best_score_:.3f} | {seconds:.1f} seconds")

Best settings: {'subsample': 0.85, 'reg_lambda': 10, 'n_estimators': 200, 'min_child_weight': 6, 'max_depth': 2, 'learning_rate': 0.1, 'colsample_bytree': 0.5}
Cross-validated AUC 0.719 | 37.8 seconds


## Score the validation and test photos

In [6]:
val_score = model.predict_proba(val[FEATURE_COLS])[:, 1]
test_score = model.predict_proba(test[FEATURE_COLS])[:, 1]

## How to read the results

- **AUC:** how well the model separates referable from non-referable photos. 1 is perfect, and 0.5 is a coin toss.
- **Sensitivity:** of the referable photos, how many it referred.
- **Specificity:** of the non-referable photos, how many it cleared.
- **95% CI:** the likely range of the true result.

The cut-off comes from validation photos only. The test set does not influence any choice here.

In [7]:
result = report("XGBoost", val_score, test_score, seconds, {k: (float(v) if isinstance(v, np.floating) else v) for k, v in search.best_params_.items()})

IndexError: index 0 is out of bounds for axis 0 with size 0

## Charts

In [ ]:
result_charts(result["model"], test_score, result["threshold"], PURPLE)

## Which features mattered?

These show what the model used. They do not prove what causes the disease.

In [ ]:
importance_chart(model.feature_importances_, "Most important XGBoost features", PURPLE)

## Save the results

In [ ]:
save_all(result, model, val_score, test_score, "xgboost")

**Next:** run the other model notebooks, then `07_compare_models.ipynb`.